## 0. Importamos las librerias que vamos a utilizar

In [181]:
import pandas as pd
import numpy as np

### 0.1 Importamos los dataframes para limpiarlos

In [182]:
deuda = pd.read_csv('../Data/1_interim/deuda.csv')
paro = pd.read_csv('../Data/1_interim/paro.csv')
pib = pd.read_csv('../Data/1_interim/pib.csv')
poblacion = pd.read_csv('../Data/1_interim/poblacion.csv')
ppcapita = pd.read_csv('../Data/1_interim/ppcapita.csv')
retail = pd.read_csv('../Data/1_interim/retail.csv')

## 1. Limpieza

### 1.1 Deuda

In [183]:
# Eliminar columnas que no necesitamos
deuda.drop(columns = ['Deuda total (M.€).1', 'Deuda total (M.$)', 'Deuda total (M.$).1', 'Deuda Per Cápita.1'], inplace = True)

In [184]:
# Cambiar nombre de la columna Fecha a Año
deuda.rename(columns= {'Fecha' : 'Year'}, inplace = True)

# Imputar valor 2009 en los años con NaN segun se vio en limpieza
deuda['Year'] = deuda['Year'].fillna(2009)

In [185]:
# Cambiar tipo de datos

# Convertir Año a entero
deuda['Year'] = deuda['Year'].astype(int)

# Convertir Deuda total (M.€) y Deuda Per Cápita a int (formato correcto)
cols = ['Deuda total (M.€)', 'Deuda Per Cápita']

for c in cols:
    deuda[c] = (
        deuda[c]
        .str.replace('.', '', regex=False)
        .str.replace(',', '.', regex=False)
        .str.replace('€', '', regex=False)
    )
    deuda[c] = deuda[c].astype(int)

# Convertir Deuda (%PIB) a float (formato correcto)

deuda['Deuda (%PIB)'] = (
    deuda['Deuda (%PIB)']
    .str.replace(',','.', regex = False)
    .str.replace('%', '', regex = False)
    .astype(float)
    .round(2)
)


In [186]:
# Arreglar columna de Países
deuda.rename(columns= {'Países' : 'Country'}, inplace = True)
deuda['Country'] = deuda['Country'].str.replace(' [+]', '')

pd.set_option('display.max_rows', None)
deuda['Country'].value_counts()

deuda.info()

<class 'pandas.DataFrame'>
RangeIndex: 572 entries, 0 to 571
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Country            572 non-null    str    
 1   Deuda total (M.€)  572 non-null    int64  
 2   Deuda (%PIB)       572 non-null    float64
 3   Deuda Per Cápita   572 non-null    int64  
 4   Year               572 non-null    int64  
dtypes: float64(1), int64(3), str(1)
memory usage: 22.5 KB


### 1.2 Paro

In [187]:
# Eliminar columnas que no necesitamos
paro.drop(columns = ['Tasa de desempleo.1', 'Var.', 'Var. Año'], inplace = True)

In [188]:
# Cambiar nombre de la columna Fecha a Año
paro.rename(columns= {'Mes' : 'Year'}, inplace = True)

paro['Year'] = paro['Year'].str.replace('Diciembre ', '')
paro['Year'].value_counts()

Year
2010                                                45
2011                                                45
2009                                                44
< Tasa de desempleo 2008Tasa de desempleo 2010 >     1
< Tasa de desempleo 2009Tasa de desempleo 2011 >     1
< Tasa de desempleo 2010Tasa de desempleo 2012 >     1
Name: count, dtype: int64

In [189]:
# Cambiar tipo de datos
paro.head()

# Cambiar a float la Tasa de desempleo
paro['Tasa de desempleo'] = (
    paro['Tasa de desempleo']
    .str.replace(',', '.')
    .str.replace('%', '')
)

# Usando errors coerce convertimos en nan todo lo que no se puede convertir en numerico, gracias a pd.to_numeric. Astype() no puede hacerlo
paro['Tasa de desempleo'] = pd.to_numeric(paro['Tasa de desempleo'], errors='coerce')

# Convertir año en float, para usar errors = coerce
paro['Year'] = pd.to_numeric(paro['Year'], errors='coerce')

# Borrar las filas que scrapearon valores incorrectos
paro.info()
paro = paro.dropna() # Conseguimos eliminar esos 3 valores gracias errors = coerce que convirtio en nulos los valores que no cumplían con el tipo

# Convertir Year en int
paro['Year'] = paro['Year'].astype(int)

<class 'pandas.DataFrame'>
RangeIndex: 137 entries, 0 to 136
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Países             137 non-null    str    
 1   Tasa de desempleo  134 non-null    float64
 2   Year               134 non-null    float64
dtypes: float64(2), str(1)
memory usage: 3.3 KB


In [190]:
# Arreglar columna de Países
paro.rename(columns= {'Países' : 'Country'}, inplace = True)
paro['Country'] = paro['Country'].str.replace(' [+]', '')

paro['Country'].value_counts()

paro.info()

<class 'pandas.DataFrame'>
Index: 134 entries, 0 to 135
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Country            134 non-null    str    
 1   Tasa de desempleo  134 non-null    float64
 2   Year               134 non-null    int64  
dtypes: float64(1), int64(1), str(1)
memory usage: 4.2 KB


### 1.3 PIB

In [191]:
# Eliminar columnas que no necesitamos
pib.drop(columns = ['PIB anual.1', 'PIB anual.2', 'PIB anual.3', 'Var. PIB (%)', 'Var. PIB (%).1'], inplace = True)

# Cambiar nombre a la columna Fecha
pib.rename(columns = {'Fecha': 'Year'}, inplace= True)
pib.rename(columns = {'Países': 'Country'}, inplace= True)


In [192]:
# Cambiar el tipo de datos y corregir formato
## 1º Cambiar el nombre de la columna
pib = pib.rename(columns = {'PIB anual' : 'PIB anual (M€)'})

pib['PIB anual (M€)'].head().apply(repr) # repr(x) devuelve la representación literal del objeto tal como Python lo ve internamente.

## 2º Cambiar el valor de los datos
pib['PIB anual (M€)'] = (
    pib['PIB anual (M€)']
    .str.replace('\xa0', '', regex=False)     # elimina el espacio raro
    .str.replace('.', '', regex=False)        # elimina puntos de miles
    .str.replace('M€', '', regex=False)       # elimina la unidad
    .str.strip()                              # elimina espacios normales
)

pib['Country'] = pib['Country'].str.replace(' [+]', '', regex = False)

# 3º Forzar com error = coerce para luego eliminar las filas mal scrapeadas
pib['PIB anual (M€)'] = pd.to_numeric(pib['PIB anual (M€)'], errors='coerce')

# 4º Eliminar filas mal scrapeadas
pib = pib.dropna()

# 5º Cambiar tipo de datos
pib['PIB anual (M€)'] = pib['PIB anual (M€)'].astype(int)
pib['Year'] = pib['Year'].astype(int)



In [193]:
pib.info()

<class 'pandas.DataFrame'>
Index: 591 entries, 0 to 592
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Country         591 non-null    str  
 1   Year            591 non-null    int64
 2   PIB anual (M€)  591 non-null    int64
dtypes: int64(2), str(1)
memory usage: 18.5 KB


### 1.4 Poblacion

In [194]:
poblacion.head()

# Eliminar columnas
poblacion.drop(columns = ['Densidad', 'Población.1', 'Var.'], inplace = True)
poblacion.rename(columns = {'Año': 'Year'}, inplace= True)

In [195]:
# Cambiar tipo de datos y arreglar formato
poblacion['Población'] = (
    poblacion['Población']
    .str.replace('.', '')
    .astype(int)
)

# Arreglar paises
poblacion.rename(columns = {'Países': 'Country'}, inplace= True)
poblacion['Country'] = poblacion['Country'].str.replace(' [+]','')
poblacion.head()


,Country,Población,Year
0,España,46486621,2009
1,Alemania,81802257,2009
2,Reino Unido,62510197,2009
3,Francia,64658856,2009
4,Italia,59690316,2009


In [196]:
poblacion.info()

<class 'pandas.DataFrame'>
RangeIndex: 588 entries, 0 to 587
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   Country    588 non-null    str  
 1   Población  588 non-null    int64
 2   Year       588 non-null    int64
dtypes: int64(2), str(1)
memory usage: 13.9 KB


### 1.5 PIB Per Capita

In [197]:
# Eliminar columnas que no necesitamos
ppcapita.drop(columns = ['PIB Per Capita.1', 'PIB Per Capita.2', 'PIB Per Capita.3', 'Var. anual PIB Per Capita', 'Var. anual PIB Per Capita.1','Var. anual PIB Per Capita.2','Var. anual PIB Per Capita.3'], inplace = True)

In [198]:
# Renombrar columnas
ppcapita.rename(columns={
    'Fecha': 'Year',
    'PIB Per Capita' : 'PIB Per Capita (€)',
    'Países' : 'Country'
}, inplace = True)

In [199]:
# Cambiar formato de los datos para poder cambiar su tipo de datos
ppcapita['PIB Per Capita (€)'] = (
    ppcapita['PIB Per Capita (€)']
    .str.replace('\xa0', '', regex=False)  
    .str.replace('.','', regex = False)
    .str.replace('€', '', regex = False)
)

ppcapita['Country'] = ppcapita['Country'].str.replace(' [+]','', regex = False)


# Cambiar tipo de datos con coerce para eliminar filas mal scrapeadas como nan
ppcapita['PIB Per Capita (€)'] = pd.to_numeric(ppcapita['PIB Per Capita (€)'], errors='coerce')
ppcapita.dropna(inplace=True)

# Cambiar tipo de datos
ppcapita[['Year', 'PIB Per Capita (€)']] = ppcapita[['Year', 'PIB Per Capita (€)']].astype(int)

ppcapita.head()


,Country,Year,PIB Per Capita (€)
0,España,2009,23140
1,Alemania,2009,30990
2,Reino Unido,2009,27920
3,Francia,2009,30040
4,Italia,2009,26600


### 1.6 Retail

In [200]:
# Cambiar tipo de datos de Customer ID a int64
retail['Customer ID'] = retail['Customer ID'].astype('Int64')

# Calcular % de ventas que salvamos al mantener nan en Customer ID
total_ventas = len(retail)
ventas_con_cliente = retail['Customer ID'].notna().sum()
ventas_sin_cliente = retail['Customer ID'].isna().sum()

porcentaje_recuperado = (ventas_sin_cliente * 100) / total_ventas

porcentaje_recuperado.round(2) # Esto nos permite salvar el 22,77% de las ventas

# Decidimos mantener los valores NaN en 'Customer ID' porque representan ventas reales
# sin cliente identificado. Sustituirlos por 0 crearía un cliente ficticio y distorsionaría
# análisis como segmentaciones, métricas por cliente o relaciones entre tablas.
# Usamos el tipo entero 'Int64', que permite NaN, para conservar la integridad del dato
# sin eliminar filas de ventas válidas ni introducir IDs artificiales.




# Imputar valor 'No description' en columna 'Description' sin descripcion
retail['Description'] = retail['Description'].fillna('No description')

In [201]:
# Convertimos 'InvoiceDate' a formato datetime para análisis temporales

retail['InvoiceDate'] = pd.to_datetime(retail['InvoiceDate'])

# Creamos una jerarquía de fechas que nos permitirá realizar análisis temporales
# detallados en el EDA y en el futuro dashboard:

retail['Year'] = retail['InvoiceDate'].dt.year # - Year: año de la transacción
retail['Month'] = retail['InvoiceDate'].dt.month # - Month: mes de la transacción
retail['Day'] = retail['InvoiceDate'].dt.day # - Day: día del mes
retail['Hour'] = retail['InvoiceDate'].dt.hour # - Hour: hora de la compra (útil para detectar patrones horarios)
retail['Week'] = retail['InvoiceDate'].dt.isocalendar().week # - Week: número de semana ISO (útil para análisis semanales)
retail['YearMonth'] = retail['InvoiceDate'].dt.to_period('M') # - YearMonth: período 'Año-Mes' para agrupar ventas mensuales de forma ordenada sin mezclar meses de distintos años y evitando problemas de ordenación.

retail.head()


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Year,Month,Day,Hour,Week,YearMonth
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,2009,12,1,7,49,2009-12
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,2009,12,1,7,49,2009-12
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,2009,12,1,7,49,2009-12
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,2009,12,1,7,49,2009-12
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,2009,12,1,7,49,2009-12


In [202]:
# Eliminar duplicados
retail.drop_duplicates(inplace=True)
retail.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Year,Month,Day,Hour,Week,YearMonth
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,2009,12,1,7,49,2009-12
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,2009,12,1,7,49,2009-12
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,2009,12,1,7,49,2009-12
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,2009,12,1,7,49,2009-12
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,2009,12,1,7,49,2009-12


In [203]:
# Crear columna isCancelled
retail['isCancelled'] = retail['Invoice'].str.startswith('C')
retail.sample(15)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Year,Month,Day,Hour,Week,YearMonth,isCancelled
69475,495543,84993a,75 GREEN PETIT FOUR CASES,1,2010-01-25 15:42:00,0.85,<NA>,United Kingdom,2010,1,25,15,4,2010-01,False
886925,568346,22811,SET OF 6 T-LIGHTS CACTI,1,2011-09-26 15:28:00,5.79,14096,United Kingdom,2011,9,26,15,39,2011-09,False
1021152,578317,23569,TRADTIONAL ALPHABET STAMP SET,4,2011-11-23 16:47:00,4.95,15398,United Kingdom,2011,11,23,16,47,2011-11,False
249386,513542,21564,PINK HEART SHAPE LOVE BUCKET,5,2010-06-25 11:57:00,2.95,16779,United Kingdom,2010,6,25,11,25,2010-06,False
703027,552134,22193,RED DINER WALL CLOCK,4,2011-05-06 11:50:00,8.50,14944,United Kingdom,2011,5,6,11,18,2011-05,False
720890,553718,22537,MAGIC DRAWING SLATE DINOSAUR,8,2011-05-18 16:14:00,0.83,<NA>,United Kingdom,2011,5,18,16,20,2011-05,False
126926,501523,21210,SET OF 72 RETRO SPOT PAPER DOILIES,12,2010-03-17 12:14:00,1.45,13612,United Kingdom,2010,3,17,12,11,2010-03,False
721598,553828,21673,WHITE SPOT BLUE CERAMIC DRAWER KNOB,24,2011-05-19 11:24:00,1.25,17865,United Kingdom,2011,5,19,11,20,2011-05,False
582128,541107,35651,VINTAGE BEAD PINK SCARF,12,2011-01-13 14:55:00,1.65,17744,United Kingdom,2011,1,13,14,2,2011-01,False
59822,494782,21379,CAMPHOR WOOD PORTOBELLO MUSHROOM,1,2010-01-18 13:15:00,2.51,<NA>,United Kingdom,2010,1,18,13,3,2010-01,False


In [204]:
retail['Description'].value_counts()

Description
WHITE HANGING HEART T-LIGHT HOLDER     5740
REGENCY CAKESTAND 3 TIER               4295
No description                         4275
JUMBO BAG RED RETROSPOT                3388
ASSORTED COLOUR BIRD ORNAMENT          2868
PARTY BUNTING                          2730
STRAWBERRY CERAMIC TRINKET BOX         2534
LUNCH BAG  BLACK SKULL.                2447
JUMBO STORAGE BAG SUKI                 2387
JUMBO SHOPPER VINTAGE RED PAISLEY      2232
HEART OF WICKER SMALL                  2219
60 TEATIME FAIRY CAKE CASES            2193
LUNCH BAG SPACEBOY DESIGN              2149
BAKING SET 9 PIECE RETROSPOT           2135
LUNCH BAG CARS BLUE                    2134
WOODEN FRAME ANTIQUE WHITE             2125
HOME BUILDING BLOCK WORD               2107
NATURAL SLATE HEART CHALKBOARD         2090
PAPER CHAIN KIT 50'S CHRISTMAS         2084
POSTAGE                                2079
WOODEN PICTURE FRAME WHITE FINISH      2056
PACK OF 60 PINK PAISLEY CAKE CASES     2036
HEART OF WICKER LARG

In [205]:
# Estandarizar la columna Description a minúsculas y limpiar espacios
retail['Description'] = (
    retail['Description']
    .str.lower()
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

In [206]:
# Solo nos interesan las ventas realizadas y canceladas, no los movimientos internos del almacen
# Por ello todas aquellas lineas que tengan de precio 0 y cantidad 0 no son ventas y debemos filtrar para que no aparezcan

# 2. Filtrar ventas reales
ventas_reales = retail[
    (retail['Price'] != 0) &                 # Precio mayor que 0
    (retail['Quantity'] != 0) &              # Cantidad mayor que 0
    (retail['Customer ID'].notna()) &        # Cliente identificado
    (~retail['StockCode'].isin(['POST', 'M']))  # Excluir envíos (POST) y ajustes manuales (M)
]

ventas_reales.info()


<class 'pandas.DataFrame'>
Index: 794754 entries, 0 to 1067369
Data columns (total 15 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      794754 non-null  str           
 1   StockCode    794754 non-null  str           
 2   Description  794754 non-null  str           
 3   Quantity     794754 non-null  int64         
 4   InvoiceDate  794754 non-null  datetime64[us]
 5   Price        794754 non-null  float64       
 6   Customer ID  794754 non-null  Int64         
 7   Country      794754 non-null  str           
 8   Year         794754 non-null  int32         
 9   Month        794754 non-null  int32         
 10  Day          794754 non-null  int32         
 11  Hour         794754 non-null  int32         
 12  Week         794754 non-null  UInt32        
 13  YearMonth    794754 non-null  period[M]     
 14  isCancelled  794754 non-null  bool          
dtypes: Int64(1), UInt32(1), bool(1), datetime64[us](1

##  2. Traducir paises de ventas_reales para hacer un posterior merge correcto

In [207]:
pd.set_option('display.max_rows', None)
list(ventas_reales['Country'].unique())

['United Kingdom',
 'France',
 'Australia',
 'EIRE',
 'Germany',
 'Portugal',
 'Japan',
 'Denmark',
 'Netherlands',
 'Poland',
 'Spain',
 'Channel Islands',
 'Italy',
 'Cyprus',
 'Belgium',
 'Greece',
 'Norway',
 'Austria',
 'Sweden',
 'United Arab Emirates',
 'Finland',
 'Switzerland',
 'USA',
 'Unspecified',
 'Nigeria',
 'Malta',
 'RSA',
 'Singapore',
 'Bahrain',
 'Thailand',
 'Israel',
 'Lithuania',
 'West Indies',
 'Korea',
 'Brazil',
 'Canada',
 'Iceland',
 'Lebanon',
 'Saudi Arabia',
 'Czech Republic',
 'European Community']

In [208]:
# 1. Traducir ventas_reales con paro
traduccion_paises = {
    'España': 'Spain',
    'Alemania': 'Germany',
    'Reino Unido': 'United Kingdom',
    'Francia': 'France',
    'Italia': 'Italy',
    'Portugal': 'Portugal',
    'Estados Unidos': 'USA',
    'Japón': 'Japan',
    'Australia': 'Australia',
    'Bélgica': 'Belgium',
    'Canadá': 'Canada',
    'Dinamarca': 'Denmark',
    'Países Bajos': 'Netherlands',
    'Polonia': 'Poland',
    'Chipre': 'Cyprus',
    'Grecia': 'Greece',
    'Noruega': 'Norway',
    'Austria': 'Austria',
    'Suecia': 'Sweden',
    'Emiratos Árabes Unidos': 'United Arab Emirates',
    'Finlandia': 'Finland',
    'Suiza': 'Switzerland',
    'Nigeria': 'Nigeria',
    'Malta': 'Malta',
    'Singapur': 'Singapore',
    'Baréin': 'Bahrain',
    'Tailandia': 'Thailand',
    'Israel': 'Israel',
    'Lituania': 'Lithuania',
    'Brasil': 'Brazil',
    'Islandia': 'Iceland',
    'Líbano': 'Lebanon',
    'Arabia Saudita': 'Saudi Arabia',
    'Chequia': 'Czech Republic',
    
    # Casos especiales
    'Irlanda': 'EIRE',
    'Sudáfrica': 'RSA',
    'Corea del Sur': 'Korea'
}

# Traduciendo los valores

for df in [deuda, paro, pib, poblacion, ppcapita]:
    df['Country'] = df['Country'].replace(traduccion_paises)

ppcapita.sample(20)


,Country,Year,PIB Per Capita (€)
418,Barbados,2011,14246
85,Irak,2009,2657
384,Uruguay,2010,9756
200,United Kingdom,2010,29830
381,Tanzania,2010,560
172,Togo,2009,517
93,Kiribati,2009,1012
63,Fiyi,2009,2643
306,Luxemburgo,2010,83550
139,Panamá,2009,5541


## 3. Exportar dataframes a la carpeta 2_cleaned

In [209]:
lista = [
    ('deuda', deuda),
    ('paro', paro),
    ('pib', pib),
    ('poblacion', poblacion),
    ('ppcapita', ppcapita),
    ('ventas', ventas_reales)
]

for nombre, df in lista:
    df.to_csv(f"../Data/2_cleaned/{nombre}.csv", index=False)